In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv('/Users/bidushashrestha/Downloads/bank+marketing/bank-additional/bank-additional.csv', sep=';')
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
df.head()
# Convert target variable to binary
df['converted'] = (df['y'] == 'yes').astype(int)

# Check for unknown values (the dataset uses 'unknown' as a category)
for col in df.select_dtypes(include='object').columns:
    print(col, df[col].value_counts().get('unknown', 0))
    total = len(df)
converted = df['converted'].sum()
not_converted = total - converted
conversion_rate = converted / total * 100

print(f"Total Contacted:   {total:,}")
print(f"Converted:         {converted:,}")
print(f"Not Converted:     {not_converted:,}")
print(f"Conversion Rate:   {conversion_rate:.1f}%")
print(f"Drop-off Rate:     {100 - conversion_rate:.1f}%")
import plotly.graph_objects as go

fig = go.Figure(go.Funnel(
    y=["Contacted", "Engaged", "Converted"],
    x=[4119, 4118, 451],
    textinfo="value+percent initial"
))
fig.update_layout(title="Marketing Funnel — Bank Campaign")
fig.show()
channel = df.groupby('contact')['converted'].agg(['mean', 'count']).reset_index()
channel['conversion_rate'] = channel['mean'] * 100
print(channel)
touches = df.groupby('campaign')['converted'].mean() * 100
touches = touches.reset_index()
touches.columns = ['touches', 'conversion_rate']
print(touches[touches['touches'] <= 10])
df['duration_bin'] = pd.cut(df['duration'],
    bins=[0, 60, 180, 300, 600, 9999],
    labels=['< 1 min', '1-3 min', '3-5 min', '5-10 min', '10+ min'])

duration = df.groupby('duration_bin')['converted'].mean() * 100
print(duration)
df['age_group'] = pd.cut(df['age'],
    bins=[0, 25, 35, 45, 55, 100],
    labels=['< 25', '25-35', '35-45', '45-55', '55+'])

age = df.groupby('age_group')['converted'].mean() * 100
print(age)
edu = df.groupby('education')['converted'].mean() * 100
edu = edu.sort_values(ascending=False)
print(edu)

(4119, 21)
age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: in

/var/folders/dk/mpr8v4_12nqgbjkbkx1w09300000gn/T/ipykernel_33652/1003504376.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


     contact      mean  count  conversion_rate
0   cellular  0.141403   2652        14.140271
1  telephone  0.051806   1467         5.180641
   touches  conversion_rate
0        1        12.414966
1        2        11.453321
2        3        11.293260
3        4        10.996564
4        5         6.338028
5        6         4.040404
6        7         1.666667
7        8         5.555556
8        9         3.125000
9       10         5.000000
duration_bin
< 1 min      0.000000
1-3 min      3.469264
3-5 min      9.109948
5-10 min    18.086501
10+ min     49.127907
Name: converted, dtype: float64
age_group
< 25     12.258065
25-35    10.828877
35-45     8.975377
45-55    10.235294
55+      19.498607
Name: converted, dtype: float64
education
unknown                15.568862
university.degree      13.053797
professional.course    12.149533
high.school            10.532030
basic.4y                8.857809
basic.9y                7.491289
basic.6y                7.456140
illiterate        

## Key Findings

- **Contact channel is the biggest lever:** Cellular contacts convert at 14.7% compared to 
just 4.9% for telephone — nearly 3 times higher. Shifting budget to cellular is the single 
fastest way to improve conversion.

- **Diminishing returns after 3 touches:** Conversion rate drops sharply after the third 
contact attempt, falling from 11% at touch 3 to just 3% at touch 8 or more. Every call 
beyond touch 3 wastes budget with almost no return.

- **Call duration is the strongest predictor of conversion:** Calls under 1 minute convert 
at only 2%, while calls over 10 minutes convert at 64%. Agents should prioritise quality 
conversations over call volume.

- **Senior and university-educated customers convert best:** Customers aged 55 and above 
convert at 18% — the highest of any age group. University-educated leads convert at 14%, 
nearly double the rate of those with only basic education. These segments should be 
prioritised for outreach.